# Agent-Environment Interface

In [8]:
!pip install gymnasium

import gymnasium as gym
from gymnasium import spaces
import numpy as np

# OR if you want to use the old name for compatibility:
import gymnasium as gym
gym = gym  # This maintains the 'gym' namespace
# Gymnasium's fundamental interface
class TradingEnv(gym.Env):
    def __init__(self):
        super().__init__()
        # Define observation and action spaces
    
    def reset(self):
        """Reset environment to initial state"""
        observation = self._get_observation()
        info = {}
        return observation, info
    
    def step(self, action):
        """Execute one timestep"""
        # 1. Apply action
        # 2. Calculate reward
        # 3. Get next observation
        # 4. Check if done
        # 5. Return step information
        return observation, reward, terminated, truncated, info

# State Space (𝒮) Design - Deep Dive

In [11]:
class State:
    """Trading state representation"""
    
    def __init__(self):
        # A. Market Microstructure Features
        self.order_book = {
            'bid_prices': [],      # Top k bid levels
            'bid_volumes': [],     # Corresponding volumes
            'ask_prices': [],      # Top k ask levels  
            'ask_volumes': [],     # Corresponding volumes
            'mid_price': float,    # (bid + ask)/2
            'spread': float,       # ask - bid
            'vwap': float,         # Volume weighted average price
        }
        
        # B. Technical Indicators (window-based)
        self.technical_features = {
            'returns': [],         # Recent returns (1m, 5m, 15m)
            'volatility': [],      # Realized volatility measures
            'volume_profile': [],  # Volume relative to average
            'momentum': [],        # Price momentum indicators
            'support_resistance': [], # Key price levels
        }
        
        # C. Portfolio State
        self.portfolio = {
            'cash': float,         # Available cash
            'position': float,     # Current holdings (+/-)
            'inventory': float,    # Absolute position size
            'unrealized_pnl': float,
            'realized_pnl': float,
            'portfolio_value': float,
            'margin_used': float,  # For leveraged trading
            'position_limit': float, # Maximum allowed position
        }
        
        # D. Market Context
        self.context = {
            'time_of_day': float,  # Normalized [0,1]
            'day_of_week': int,    # 0-4 for weekdays
            'volatility_regime': int, # High/Medium/Low
            'liquidity_regime': int,  # High/Low liquidity
            'market_state': int,   # Trending/Ranging
        }
        
        # E. Risk Metrics
        self.risk = {
            'var_95': float,       # Value at Risk 95%
            'expected_shortfall': float,
            'sharpe_ratio': float,
            'max_drawdown': float,
            'correlation_matrix': np.array, # To other instruments
        }

# Practical Implementation Considerations:

In [15]:
def normalize_state(state_dict):
    """Normalize state components for neural network consumption"""
    normalized = {}
    
    # Z-score normalization for continuous variables
    for key in ['prices', 'volumes', 'returns']:
        mean, std = get_running_stats(key)
        normalized[key] = (state_dict[key] - mean) / (std + 1e-8)
    
    # Min-max normalization for bounded variables
    for key in ['time_of_day', 'position_ratio']:
        min_val, max_val = get_bounds(key)
        normalized[key] = (state_dict[key] - min_val) / (max_val - min_val)
    
    # One-hot encoding for categorical variables
    normalized['day_encoding'] = one_hot(state_dict['day_of_week'], 5)
    
    return normalized

# Action Space (𝒜) - Detailed Taxonomy
## Discrete Action Space Example:

In [19]:
import gym
from gymnasium import spaces

class DiscreteTradingEnv(gym.Env):
    def __init__(self):
        # Simple discrete actions
        self.action_space = spaces.Discrete(5)
        # Actions: 0=Hold, 1=Buy 1 unit, 2=Sell 1 unit, 
        #          3=Buy 0.5 units, 4=Sell 0.5 units

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Continuous Action Space Example:

In [22]:
class ContinuousTradingEnv(gym.Env):
    def __init__(self):
        # Multi-dimensional continuous actions
        self.action_space = spaces.Box(
            low=np.array([-1.0, 0.0, 0.0]),   # [direction, size, aggressiveness]
            high=np.array([1.0, 10.0, 1.0]),  # Upper bounds
            dtype=np.float32
        )
        # action[0]: -1 (sell) to +1 (buy)
        # action[1]: order size (0 to max_position)
        # action[2]: execution aggressiveness (0=passive to 1=aggressive)

## Hybrid Action Space (Best of Both Worlds):

In [25]:
class HybridTradingEnv(gym.Env):
    def __init__(self):
        # Discrete decision + continuous parameters
        self.action_space = spaces.Dict({
            'order_type': spaces.Discrete(3),  # 0=Market, 1=Limit, 2=Cancel
            'direction': spaces.Discrete(3),   # 0=Sell, 1=Hold, 2=Buy
            'quantity': spaces.Box(low=0, high=100, shape=(1,)),
            'limit_price': spaces.Box(low=0, high=np.inf, shape=(1,))
        })

# Reward Function (R) - Complete Formulation
## Basic Reward - Immediate PnL:

In [28]:
def simple_pnl_reward(old_portfolio_value, new_portfolio_value):
    """Simple PnL-based reward"""
    return new_portfolio_value - old_portfolio_value

## Sophisticated Trading Reward Functions:

In [31]:
class TradingReward:
    def __init__(self, config):
        self.config = config
        self.transaction_cost_rate = 0.001  # 10bps
        
    def calculate_reward(self, state, action, next_state):
        """Comprehensive reward calculation"""
        
        # 1. Base PnL component
        pnl = next_state.portfolio_value - state.portfolio_value
        
        # 2. Transaction cost penalty
        transaction_cost = self._calculate_transaction_cost(state, action)
        
        # 3. Inventory risk penalty (encourage mean reversion)
        inventory_penalty = -self.config.inventory_weight * (
            next_state.position ** 2
        )
        
        # 4. Drawdown penalty
        drawdown_penalty = -self._calculate_drawdown_penalty(state, next_state)
        
        # 5. Risk-adjusted return component
        sharpe_component = self._calculate_sharpe_contribution(state, next_state)
        
        # 6. Slippage penalty (for large orders)
        slippage_penalty = -self._calculate_slippage(action, state.market_depth)
        
        # 7. Opportunity cost penalty (for missed trades)
        opportunity_penalty = -self._calculate_opportunity_cost(state, action)
        
        # Combined reward
        total_reward = (
            self.config.pnl_weight * pnl +
            self.config.transaction_weight * (-transaction_cost) +
            inventory_penalty +
            drawdown_penalty +
            self.config.sharpe_weight * sharpe_component +
            slippage_penalty +
            opportunity_penalty
        )
        
        return total_reward
    
    def _calculate_transaction_cost(self, state, action):
        """Calculate realistic transaction costs"""
        if action['order_type'] == 'market':
            # Market order: spread cost + commission
            spread_cost = state.spread * 0.5  # Pay half spread
            commission = abs(action['quantity']) * state.mid_price * self.commission_rate
            return spread_cost + commission
        elif action['order_type'] == 'limit':
            # Limit order: only commission if filled
            fill_probability = self._estimate_fill_probability(action, state)
            commission = abs(action['quantity']) * state.mid_price * self.commission_rate
            return fill_probability * commission
        return 0

# Transition Dynamics (P) - Market Reality

# Simulation-Based Learning:

In [35]:
class MarketSimulator:
    """Simulates market transitions"""
    
    def transition(self, state, action):
        """Simulate market response to action"""
        
        # 1. Apply action to market
        order_book_impact = self._apply_order(action, state.order_book)
        
        # 2. Simulate market evolution
        price_change = self._sample_price_change(state)
        volume_change = self._sample_volume_change(state)
        
        # 3. Update portfolio
        new_portfolio = self._update_portfolio(state.portfolio, action)
        
        # 4. Calculate new state
        new_state = State(
            order_book=order_book_impact,
            portfolio=new_portfolio,
            market_features=self._update_features(state, price_change)
        )
        
        return new_state
    
    def _sample_price_change(self, state):
        """Sample from empirical distribution or stochastic model"""
        # Option 1: Historical simulation
        if self.mode == 'historical':
            return self.historical_data.sample_next_return()
        
        # Option 2: Stochastic model
        elif self.mode == 'stochastic':
            # GBM with stochastic volatility
            drift = state.expected_return
            volatility = state.volatility
            return np.random.normal(drift, volatility)
        
        # Option 3: Generative model
        elif self.mode == 'generative':
            return self.gan.sample(state)

# Discount Factor (γ) - Time Preference Analysis
## Trading-Specific γ Values:

In [38]:
# Different trading styles require different discount factors
DISCOUNT_FACTORS = {
    'high_frequency': 0.95,      # Very short-term focus
    'day_trading': 0.97,         # End-of-day closeout important
    'swing_trading': 0.99,       # Multi-day holding periods
    'position_trading': 0.995,   # Long-term positions
    'market_making': 0.98,       # Inventory management focus
}

# Equivalent horizon in time steps
def effective_horizon(gamma):
    """Calculate effective planning horizon"""
    # Time until future reward is discounted to 1/e ≈ 37%
    horizon = 1 / (1 - gamma)
    return horizon

# Policy and Value Functions - Trading Interpretation
## Policy π(a|s) - The Trading Strategy

In [43]:
class TradingPolicy:
    """Maps states to trading actions"""
    
    # def __init__(self):
    #     # Could be:
    #     # 1. Neural network (Deep RL)
    #     # 2. Lookup table (Tabular RL)
    #     # 3. Rule-based (Baseline)
        
    def act(self, state):
        """Select action given state"""
        # Exploration vs exploitation trade-off
        if np.random.random() < self.epsilon:
            return self._explore(state)  # Try random action
        else:
            return self._exploit(state)  # Use learned policy

# Trading Example:

In [48]:
import numpy as np
import torch
import torch.nn as nn
from typing import List, Tuple

# ============================================================================
# 1. DEFINE THE TRADING ENVIRONMENT
# ============================================================================

class TradingEnvironment:
    """Simple trading environment for demonstration"""
    
    def __init__(self, data_length: int = 100):
        self.data_length = data_length
        self.current_step = 0
        
        # Generate synthetic market data
        self.prices = 100 + np.cumsum(np.random.randn(data_length) * 0.1)
        self.volumes = 1000 + np.random.randn(data_length) * 100
        
        # Initialize portfolio
        self.cash = 10000.0
        self.position = 0
        self.initial_cash = self.cash
        
        # Define available actions
        self.available_actions = ['hold', 'buy', 'sell']
        
    def reset(self):
        """Reset environment to initial state"""
        self.current_step = 0
        self.cash = self.initial_cash
        self.position = 0
        return self.get_state()
    
    def get_state(self) -> dict:
        """Get current state observation"""
        current_price = self.prices[self.current_step]
        current_volume = self.volumes[self.current_step]
        
        # Calculate simple technical indicators
        if self.current_step >= 4:
            recent_prices = self.prices[self.current_step-4:self.current_step+1]
            returns = np.diff(recent_prices) / recent_prices[:-1]
            volatility = np.std(returns) if len(returns) > 0 else 0
        else:
            volatility = 0
        
        return {
            'price': current_price,
            'volume': current_volume,
            'returns': (current_price / self.prices[max(0, self.current_step-1)]) - 1,
            'volatility': volatility,
            'position': self.position,
            'cash': self.cash,
            'portfolio_value': self.cash + (self.position * current_price),
            'step': self.current_step / self.data_length,  # Normalized time
        }
    
    def get_available_actions(self) -> List[str]:
        """Get list of available actions in current state"""
        return self.available_actions
    
    def _action_to_index(self, action: str) -> int:
        """Convert action string to index"""
        action_map = {'hold': 0, 'buy': 1, 'sell': 2}
        return action_map.get(action, 0)
    
    def step(self, action: str) -> Tuple[dict, float, bool]:
        """Execute action and return next state, reward, done"""
        current_price = self.prices[self.current_step]
        
        # Execute action
        if action == 'buy' and self.cash >= current_price:
            self.position += 1
            self.cash -= current_price
            transaction_cost = current_price * 0.001  # 0.1% transaction cost
            self.cash -= transaction_cost
        elif action == 'sell' and self.position > 0:
            self.position -= 1
            self.cash += current_price
            transaction_cost = current_price * 0.001  # 0.1% transaction cost
            self.cash -= transaction_cost
        
        # Move to next time step
        self.current_step += 1
        
        # Calculate reward (change in portfolio value)
        old_value = self._calculate_portfolio_value(self.current_step - 1)
        new_value = self._calculate_portfolio_value(self.current_step)
        reward = new_value - old_value
        
        # Check if episode is done
        done = (self.current_step >= self.data_length - 1 or 
                self.cash <= 0 or
                new_value <= self.initial_cash * 0.5)  # 50% drawdown limit
        
        return self.get_state(), reward, done
    
    def _calculate_portfolio_value(self, step: int) -> float:
        """Calculate portfolio value at specific step"""
        if step < 0 or step >= len(self.prices):
            return self.cash
        return self.cash + (self.position * self.prices[step])
    
    def render(self):
        """Print current state"""
        state = self.get_state()
        print(f"Step {self.current_step}: "
              f"Price=${state['price']:.2f}, "
              f"Position={state['position']}, "
              f"Cash=${state['cash']:.2f}, "
              f"Value=${state['portfolio_value']:.2f}")

# ============================================================================
# 2. DEFINE THE Q-NETWORK MODEL
# ============================================================================

class DQN(nn.Module):
    """Deep Q-Network for trading decisions"""
    
    def __init__(self, input_dim: int, output_dim: int):
        super(DQN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

# ============================================================================
# 3. STATE NORMALIZATION FUNCTION
# ============================================================================

def normalize_state(state: dict) -> np.ndarray:
    """Normalize state features for neural network input"""
    # Extract and normalize features
    normalized_features = np.array([
        state['price'] / 150.0,  # Normalize price (assuming max ~150)
        state['volume'] / 1500.0,  # Normalize volume
        state['returns'] * 10,  # Scale returns
        state['volatility'] * 10,  # Scale volatility
        state['position'] / 10.0,  # Normalize position
        state['cash'] / 20000.0,  # Normalize cash
        state['step']  # Already normalized [0, 1]
    ], dtype=np.float32)
    
    return normalized_features

# ============================================================================
# 4. Q-VALUE ESTIMATION FUNCTION (FIXED VERSION)
# ============================================================================

def estimate_q_value(state: dict, action: str, model: DQN) -> float:
    """Estimate Q(s,a) for trading"""
    # Normalize state features
    normalized_features = normalize_state(state)
    
    # Convert to tensor
    state_tensor = torch.tensor(normalized_features, dtype=torch.float32).unsqueeze(0)
    
    # Deep Q-Network forward pass
    with torch.no_grad():
        q_values = model(state_tensor)  # Shape: [1, num_actions]
    
    # Convert action to index
    action_map = {'hold': 0, 'buy': 1, 'sell': 2}
    action_idx = action_map.get(action, 0)
    
    # Return Q-value for the specific action
    return q_values[0, action_idx].item()

# ============================================================================
# 5. MAIN EXECUTION - COMPLETE WORKING EXAMPLE
# ============================================================================

def main():
    print("=" * 60)
    print("TRADING RL - COMPLETE WORKING EXAMPLE")
    print("=" * 60)
    
    # ------------------------------------------------------------------------
    # Step 1: Create the environment
    # ------------------------------------------------------------------------
    print("\n1. Creating trading environment...")
    env = TradingEnvironment(data_length=50)  # <-- THIS CREATES THE 'env' OBJECT
    print("✓ Environment created successfully")
    
    # ------------------------------------------------------------------------
    # Step 2: Initialize the Q-network
    # ------------------------------------------------------------------------
    print("\n2. Initializing Q-network...")
    input_dim = 7  # Number of state features
    output_dim = 3  # Number of actions (hold, buy, sell)
    q_network = DQN(input_dim, output_dim)
    print("✓ Q-network initialized")
    
    # Load or initialize weights (here we use random weights for demonstration)
    # In a real scenario, you would load trained weights
    
    # ------------------------------------------------------------------------
    # Step 3: Reset environment and get initial state
    # ------------------------------------------------------------------------
    print("\n3. Resetting environment...")
    initial_state = env.reset()
    print(f"Initial state:")
    for key, value in initial_state.items():
        print(f"  {key}: {value}")
    
    # ------------------------------------------------------------------------
    # Step 4: Make trading decisions using Q-values
    # ------------------------------------------------------------------------
    print("\n4. Making trading decisions...")
    
    for step in range(10):  # Run for 10 steps
        print(f"\n--- Step {step} ---")
        
        # Get current state
        current_state = env.get_state()  # <-- NOW 'env' IS DEFINED
        
        # Get available actions
        possible_actions = env.get_available_actions()  # <-- NOW 'env' IS DEFINED
        
        print(f"Available actions: {possible_actions}")
        
        # Estimate Q-values for each possible action
        q_values = []
        for action in possible_actions:
            q_val = estimate_q_value(current_state, action, q_network)
            q_values.append(q_val)
            print(f"  Q({action}) = {q_val:.4f}")
        
        # Select action with highest Q-value
        best_action_idx = np.argmax(q_values)
        best_action = possible_actions[best_action_idx]
        
        print(f"Selected action: {best_action} (Q-value: {q_values[best_action_idx]:.4f})")
        
        # Execute action
        next_state, reward, done = env.step(best_action)
        
        print(f"Reward: {reward:.4f}")
        env.render()
        
        if done:
            print("\nEpisode terminated!")
            break
    
    # ------------------------------------------------------------------------
    # Step 5: Demonstrate the conceptual usage pattern
    # ------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("CONCEPTUAL USAGE PATTERN DEMONSTRATION")
    print("=" * 60)
    
    # Show how the original conceptual code would work
    env_demo = TradingEnvironment(data_length=20)
    env_demo.reset()
    
    # This is the pattern from the original example, now working:
    demo_state = env_demo.get_state()
    demo_actions = env_demo.get_available_actions()
    
    print(f"\ncurrent_state = env.get_state()")
    print(f"  → Returns: {type(demo_state)} with keys: {list(demo_state.keys())}")
    
    print(f"\npossible_actions = env.get_available_actions()")
    print(f"  → Returns: {demo_actions}")
    
    print(f"\nq_values = [estimate_q_value(current_state, a, q_network) for a in possible_actions]")
    demo_q_values = [estimate_q_value(demo_state, a, q_network) for a in demo_actions]
    print(f"  → Returns: {[f'{v:.4f}' for v in demo_q_values]}")
    
    print(f"\nbest_action = possible_actions[np.argmax(q_values)]")
    demo_best_action = demo_actions[np.argmax(demo_q_values)]
    print(f"  → Returns: '{demo_best_action}'")

# ============================================================================
# 6. ALTERNATIVE: SIMPLIFIED VERSION FOR QUICK TESTING
# ============================================================================

def quick_test():
    """Quick test to verify the fix"""
    print("\n" + "=" * 60)
    print("QUICK TEST - VERIFYING THE FIX")
    print("=" * 60)
    
    # Create environment
    env = TradingEnvironment(data_length=10)
    
    # Create a simple model
    model = DQN(input_dim=7, output_dim=3)
    
    # Get state and actions
    env.reset()
    current_state = env.get_state()
    possible_actions = env.get_available_actions()
    
    print(f"Environment created: {type(env)}")
    print(f"State retrieved: {len(current_state)} features")
    print(f"Available actions: {possible_actions}")
    
    # This should now work without errors
    q_values = [estimate_q_value(current_state, a, model) for a in possible_actions]
    best_action = possible_actions[np.argmax(q_values)]
    
    print(f"\nQ-values: {[f'{v:.4f}' for v in q_values]}")
    print(f"Best action: {best_action}")
    
    print("\n✓ Fix verified! No more 'env is not defined' error.")

# ============================================================================
# RUN THE SCRIPT
# ============================================================================

if __name__ == "__main__":
    # Set random seeds for reproducibility
    np.random.seed(42)
    torch.manual_seed(42)
    
    # Run the complete example
    main()
    
    # Also run the quick test
    quick_test()

TRADING RL - COMPLETE WORKING EXAMPLE

1. Creating trading environment...
✓ Environment created successfully

2. Initializing Q-network...
✓ Q-network initialized

3. Resetting environment...
Initial state:
  price: 100.04967141530112
  volume: 1032.4083969394794
  returns: 0.0
  volatility: 0
  position: 0
  cash: 10000.0
  portfolio_value: 10000.0
  step: 0.0

4. Making trading decisions...

--- Step 0 ---
Available actions: ['hold', 'buy', 'sell']
  Q(hold) = -0.0837
  Q(buy) = 0.1682
  Q(sell) = 0.0411
Selected action: buy (Q-value: 0.1682)
Reward: -0.0138
Step 1: Price=$100.04, Position=1, Cash=$9899.85, Value=$9999.89

--- Step 1 ---
Available actions: ['hold', 'buy', 'sell']
  Q(hold) = -0.0857
  Q(buy) = 0.1671
  Q(sell) = 0.0401
Selected action: buy (Q-value: 0.1671)
Reward: 0.1295
Step 2: Price=$100.10, Position=2, Cash=$9799.71, Value=$9999.92

--- Step 2 ---
Available actions: ['hold', 'buy', 'sell']
  Q(hold) = -0.0873
  Q(buy) = 0.1662
  Q(sell) = 0.0383
Selected action: 